In [2]:
%pip install --upgrade pip
%pip install gepa

%pip install openai
%pip install litellm

%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
# Von .env Datei einlesen:
## HF_TOKEN
## OPENAI_API_KEY
## OPENAI_API_BASE
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [2]:
def banner(title):
    print("=" * 60)
    print(f"  {title}")
    print("=" * 60)
    print()

# First Run
### Optimize a prompt for extracting numbers from sentences and adding them up

In [3]:
from openai import OpenAI
import os
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_API_BASE")
)

# Test-Cases
CASES = [
    {"input" : "I have 3 apples and 5 oranges.",         "expected" : 8 },
    {"input": "There are 12 students and 4 teachers.",   "expected" : 16},
    {"input": "She spent 20 euros and saved 15 euros.",  "expected" : 35},
    {"input": "Zero items remain.",                      "expected" : 0 },
]

# Evaluator
import re
import gepa.optimize_anything as oa

def evaluate(candidate: str) -> tuple[float, dict]:
    correct = 0
    errors  = []
    
    for case in CASES:
        response = client.chat.completions.create(
            model="Llama-3.3-70B",
            messages=[
                {"role": "system", "content" : candidate},
                {"role": "user", "content": case["input"]},
            ],
        )
        
        raw = response.choices[0].message.content.strip()
        print("Raw output of LLM:")
        banner(raw)
        
        # Expected: Anwer is only a number
        try:
            result = int(re.search(r"\d+", raw).group())
            if result == case["expected"]:
                correct +=1
            else :
                errors.append(f"'{case['input']}' was {result} but '{case['expected']}' was expected.")
        except Exception:
                errors.append(f"'{case['input']} nicht parsebar: '{raw}'")
    
    score = correct / len(CASES)
    oa.log(f"Korrekt: {correct} / {len(CASES)}")
    oa.log(f"Fehler: {errors}")
    
    print("One EVALUATE cycle completed.\n")
    
    return score, {"correct": correct, "errors": errors}


# Optimizer
from gepa.optimize_anything import optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig

result = oa.optimize_anything(
    #seed_candidate="Extract all numbers from the text an dreturn their sum.",
    seed_candidate=None,
    evaluator=evaluate,
    objective="Optimize a system primt that majes an LLM sum all numbers in a given sentence. Reply with only the number.",
    config=GEPAConfig(
        engine=EngineConfig(max_metric_calls=10),
        reflection=ReflectionConfig(reflection_lm="openai/gpt-oss-120b")
    )
)
print("RESULT cycle completed.\n")

print(result.best_candidate)

Generating initial seed candidate via LLM...
Generated seed candidate (410 chars)
Raw output of LLM:
  8

Raw output of LLM:
  16

Raw output of LLM:
  35

Raw output of LLM:
  0

One EVALUATE cycle completed.

Iteration 0: Base program full valset score: 1.0 over 1 / 1 examples
Iteration 1: Selected program 0 score: 1.0
Raw output of LLM:
  8

Raw output of LLM:
  16

Raw output of LLM:
  35

Raw output of LLM:
  0

One EVALUATE cycle completed.

Iteration 1: Proposed new text for current_candidate: You are a concise assistant whose sole task is to read the provided sentence, locate every numeric value it contains, compute the sum of those values, and output **only** the resulting sum as a plain number.

Guidelines:
- Recognize integers, decimal numbers, negative numbers, and numbers that use commas as thousand separators (e.g., 1,234 or -2,500.75). Treat commas purely as digit grouping.
- Accept numbers that appear directly before or after non‑alphabetic characters (e.g., “5kg”, “$12

In [ ]:
import gepa.optimize_anything as oa

def evaluate(candidate: str) -> float:
    score, diagnostic = run_my_system(candidate)
    oa.log(f"Error: {diagnostic}")  # captured as ASI
    return score

# Start from an existing artifact…
result = oa.optimize_anything(
    seed_candidate="<your initial artifact>",
    evaluator=evaluate,
)

# … or just describe what you need.
result = oa.optimize_anything(
    evaluator=evaluate,
    objective="Generate a Python function `reverse()` that reverses a string.",
)

print(result.best_candidate)

In [ ]:
import gepa.optimize_anything as oa

# Structured Dictionary for richer dignostics
def evaluate(candidate: str) -> tuple[float, dict]:
    result = execute_code(candidate)
    return result.score, {
        "Error": result.stderr,
        "Output": result.stdout,
        "Runtime": f"{result.time_ms:.1f}ms",
    }

# Start from an existing artifact…
result = oa.optimize_anything(
    seed_candidate="<your initial artifact>",
    evaluator=evaluate,
)

# … or just describe what you need.
result = oa.optimize_anything(
    evaluator=evaluate,
    objective="Generate a Python function `reverse()` that reverses a string.",
)

print(result.best_candidate)

In [ ]:
from gepa import Image
from demo_utils import render_image, get_vlm_score_feedback

GOAL = "a pelican riding a bicycle"
VLM = "vertex_ai/gemini-3-flash-preview"

def evaluate(candidate, example):
    """Render SVG → image, score with a VLM, return (score, side_info)."""
    image = render_image(candidate["svg_code"]) # via cairosvg
    score, feedback = get_vlm_score_feedback(VLM, image, example["criteria"]) # simple regex parser

    return score, {
        "RenderedSVG": Image(base64_data=image, media_type="image/png"),
        "Feedback": feedback,
    }

VISUAL_ASPECTS = [
    # 6 visual aspects → Pareto-efficient selection
    {"id": "overall",     "criteria": f"Rate overall quality of this SVG ({GOAL}). SCORE: X/10"},
    {"id": "anatomy",     "criteria": "Rate pelican accuracy: beak, pouch, plumage. SCORE: X/10"},
    {"id": "bicycle",     "criteria": "Rate bicycle: wheels, frame, handlebars, pedals. SCORE: X/10"},
    {"id": "composition", "criteria": "Rate how convincingly the pelican rides the bicycle. SCORE: X/10"},
    {"id": "visual",      "criteria": "Rate visual appeal, scenery, and color usage. SCORE: X/10"},
    {"id": "craft",       "criteria": "Rate SVG technical quality: shapes, layering. SCORE: X/10"},
]

In [ ]:
from gepa.optimize_anything import (
    optimize_anything, GEPAConfig, EngineConfig, ReflectionConfig,
)

result = optimize_anything(
    seed_candidate={"svg_code": open("seed.svg").read()}, # a plain white canvas
    evaluator=evaluate,
    dataset=VISUAL_ASPECTS,
    objective=f"Optimize SVG code to illustrate '{GOAL}'. Output ONLY valid SVG.",
    config=GEPAConfig(
        engine=EngineConfig(max_metric_calls=150),
        reflection=ReflectionConfig(reflection_lm=VLM),
    ),
)
print(result.best_candidate)